In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.5 MB/s eta 0:00:00


In [2]:
# 1. Install official Segment Anything library
!pip install git+https://github.com/facebookresearch/segment-anything.git

# 2. Download the SAM ViT-B checkpoint (Base model, fits on T4)
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-d7wa2xfd
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-d7wa2xfd
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done


In [3]:
import os
import re
import glob
import numpy as np
import nibabel as nib
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.distance import directed_hausdorff
from sklearn.metrics import precision_score
from ultralytics import SAM
import torchvision.transforms.functional as TF
import torch.optim as optim
import torch
import torch.nn.functional as F  # Sometimes needed for other interpolations
import random
from segment_anything import sam_model_registry
from collections import defaultdict
from tqdm import tqdm

# Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# Define Constants needed for your class
VALIDATION_SPLIT = 0.1

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda


In [4]:
class BraTSDataset(Dataset):
    def __init__(self, root_dir, transform=None, img_size=128, mode='train'):
        self.root_dir = root_dir
        self.transform = transform
        self.img_size = img_size
        self.mode = mode
        
        print(f"Scanning {root_dir} for dataset...")
        
        # --- 1. SCAN & GROUP BY EXTRACTED PATIENT ID ---
        self.patient_map = defaultdict(dict)
        files_seen = 0
        
        # Regex to find Patient ID. 
        # Priority 1: BraTS2021_xxxxx
        # Priority 2: Just a sequence of 5+ digits (e.g. 00000057)
        regex_brats = re.compile(r"(BraTS2021_\d{5})")
        regex_digits = re.compile(r"(\d{5,})")

        for root, dirs, files in os.walk(root_dir):
            for filename in files:
                if filename.endswith(('.nii', '.nii.gz')):
                    files_seen += 1
                    full_path = os.path.join(root, filename)
                    lower_name = filename.lower()
                    
                    # 1. Identify Modality
                    key_type = None
                    if 'flair' in lower_name: key_type = 'flair'
                    elif 't1ce' in lower_name: key_type = 't1ce'
                    elif 't1' in lower_name: key_type = 't1' # t1ce check must be above
                    elif 't2' in lower_name: key_type = 't2'
                    elif 'seg' in lower_name: key_type = 'seg'
                    
                    if key_type:
                        # 2. Extract Patient ID from the FULL PATH (handles folders-as-files)
                        # We search the whole path string because the ID might be in the folder name
                        match = regex_brats.search(full_path)
                        if match:
                            patient_id = match.group(1)
                        else:
                            # Fallback to simple digits if BraTS prefix missing
                            match_d = regex_digits.search(filename)
                            if match_d:
                                patient_id = match_d.group(1)
                            else:
                                continue # Can't find an ID, skip file

                        # 3. Store (Overwrite if duplicates found, keeping the last one found)
                        # Only add if file is not empty
                        if os.path.getsize(full_path) > 0:
                            self.patient_map[patient_id][key_type] = full_path

        # --- 2. FILTER INCOMPLETE PATIENTS ---
        self.valid_patients = []
        for pid, files in self.patient_map.items():
            # Check if we have all 5 required modalities
            if all(k in files for k in ['flair', 't1', 't1ce', 't2', 'seg']):
                self.valid_patients.append(pid)
        
        self.valid_patients = sorted(self.valid_patients)
        
        # --- 3. TRAIN/VAL SPLIT ---
        split_idx = int(len(self.valid_patients) * (1 - VALIDATION_SPLIT))
        if mode == 'train':
            self.valid_patients = self.valid_patients[:split_idx]
        else:
            self.valid_patients = self.valid_patients[split_idx:]
            
        print(f"[{mode.upper()}] Final count: {len(self.valid_patients)} valid patients (from {len(self.patient_map)} IDs found in {files_seen} files).")

    def __len__(self):
        return len(self.valid_patients)

    def __getitem__(self, idx):
        pid = self.valid_patients[idx]
        paths = self.patient_map[pid]
        
        try:
            flair = nib.load(paths['flair']).get_fdata()
            t1 = nib.load(paths['t1']).get_fdata()
            t1ce = nib.load(paths['t1ce']).get_fdata()
            t2 = nib.load(paths['t2']).get_fdata()
            seg = nib.load(paths['seg']).get_fdata()
        except Exception as e:
            print(f"Error loading {pid}: {e}. Skipping...")
            new_idx = random.randint(0, len(self.valid_patients) - 1)
            return self.__getitem__(new_idx)

        # EXTRACT SLICE (Middle Slice)
        slice_idx = flair.shape[2] // 2
        
        # Stack modalities: (H, W, 4)
        image = np.stack([flair[:,:,slice_idx], 
                          t1[:,:,slice_idx], 
                          t1ce[:,:,slice_idx], 
                          t2[:,:,slice_idx]], axis=-1)
        mask = seg[:,:,slice_idx]

        # PREPROCESSING
        image_t = torch.tensor(image).permute(2, 0, 1).float()
        mask_t = torch.tensor(mask).unsqueeze(0).float()
        
        image_t = TF.resize(image_t, [self.img_size, self.img_size], antialias=True)
        mask_t = TF.resize(mask_t, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        
        mean = image_t.mean(dim=(1, 2), keepdim=True)
        std = image_t.std(dim=(1, 2), keepdim=True) + 1e-8
        image_t = (image_t - mean) / std
        
        mask_t[mask_t == 4] = 3
        
        return image_t, mask_t.squeeze(0).long()

In [5]:
import os
from torch.utils.data import DataLoader

# --- Initialize Loader ---
# Updated path based on your request
DATA_ROOT = '/kaggle/input/instant-odc-ai-hackathon/Train' 

# Fallback search: robustly find the 'Train' folder inside 'INSTANT-ODC AI Hackathon'
if not os.path.exists(DATA_ROOT):
    found = False
    print("Exact path not found. Searching in /kaggle/input...")
    for root, dirs, files in os.walk('/kaggle/input'):
        # Look for the specific folder "Train" usually inside "INSTANT-ODC"
        if "Train" in dirs and "INSTANT-ODC" in root:
            DATA_ROOT = os.path.join(root, "Train")
            found = True
            break
        # Or if we just find "INSTANT-ODC AI Hackathon"
        if "INSTANT-ODC AI Hackathon" in dirs:
             DATA_ROOT = os.path.join(root, "INSTANT-ODC AI Hackathon", "Train")
             found = True
             break
    
    if not found:
        # Final fallback: just look for ANY folder named "Train"
        for root, dirs, files in os.walk('/kaggle/input'):
            if "Train" in dirs:
                DATA_ROOT = os.path.join(root, "Train")
                break

print(f"Using Data Root: {DATA_ROOT}")

# Re-initialize the dataset and loader
train_dataset = BraTSDataset(DATA_ROOT, mode='train')
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

Using Data Root: /kaggle/input/instant-odc-ai-hackathon/Train
Scanning /kaggle/input/instant-odc-ai-hackathon/Train for dataset...
[TRAIN] Final count: 825 valid patients (from 917 IDs found in 4585 files).


In [6]:
# --- Metrics Helper (Same as before) ---
def calculate_metrics(pred_mask, true_mask):
    pred_bin = (pred_mask.view(-1) > 0.5).cpu().numpy().astype(int)
    true_bin = (true_mask.view(-1) > 0).cpu().numpy().astype(int)

    # Dice
    intersection = (pred_bin * true_bin).sum()
    dice = (2. * intersection) / (pred_bin.sum() + true_bin.sum() + 1e-8)

    # Precision
    precision = precision_score(true_bin, pred_bin, zero_division=0)

    # Hausdorff
    pred_map = (pred_mask[0, 0] > 0.5).cpu().numpy()
    true_map = (true_mask[0, 0] > 0).cpu().numpy()
    
    if np.any(pred_map) and np.any(true_map):
        u = np.array(np.where(pred_map)).T
        v = np.array(np.where(true_map)).T
        hd = max(directed_hausdorff(u, v)[0], directed_hausdorff(v, u)[0])
    else:
        hd = 0.0
    return dice, precision, hd

def train_cycle(model, epochs_to_train, total_epochs_so_far):
    optimizer = torch.optim.Adam(model.model.parameters(), lr=1e-4)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs_to_train):
        model.model.train()
        epoch_loss = 0
        epoch_dice = 0
        epoch_prec = 0
        epoch_hd = 0
        steps = 0

        for i, (images, masks) in enumerate(train_loader):
            images = images.to(device)
            masks = masks.to(device)

            # ADAPTATION: Handle 4-channel input from your dataset
            # SAM needs 3 channels. We take FLAIR(0), T1CE(2), T2(3).
            # Shape becomes (Batch, 3, 128, 128)
            sam_input = images[:, [0, 2, 3], :, :]

            # PREPARE MASKS: Convert Multi-class (0,1,2,3) to Binary (0, 1) for SAM
            binary_masks = (masks > 0).float().unsqueeze(1) # (Batch, 1, 128, 128)

            # SAM Forward Pass
            outputs = model.model(sam_input)
            
            # Ultralytics SAM raw output handling
            if isinstance(outputs, list):
                 pred = outputs[0]
            else:
                 pred = outputs
            
            # Interpolate if resolution differs
            if pred.shape[-2:] != binary_masks.shape[-2:]:
                 pred = torch.nn.functional.interpolate(pred, size=binary_masks.shape[-2:], mode='bilinear')
            
            # Take first channel (assuming binary/foreground task)
            pred_single = pred[:, 0:1, :, :] 

            loss = loss_fn(pred_single, binary_masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Metrics
            d, p, h = calculate_metrics(torch.sigmoid(pred_single), masks)
            
            epoch_loss += loss.item()
            epoch_dice += d
            epoch_prec += p
            epoch_hd += h
            steps += 1

        print(f"Epoch {total_epochs_so_far + epoch + 1} | "
              f"Loss: {epoch_loss/steps:.4f} | "
              f"Dice: {epoch_dice/steps:.4f} | "
              f"Precision: {epoch_prec/steps:.4f} | "
              f"Haussdorf: {epoch_hd/steps:.2f}")

In [7]:
from tqdm import tqdm
import torch
import os

def train_sam_custom(epochs_to_add, total_epochs_so_far):
    # Optimizer (Ensure we re-initialize or keep state if desired, 
    # but here we re-init for simplicity of the function call as per previous steps)
    optimizer = optim.Adam([
        {'params': sam_model.mask_decoder.parameters()},
        {'params': sam_model.prompt_encoder.parameters()}
    ], lr=1e-5, weight_decay=1e-4)
    
    loss_fn = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs_to_add):
        sam_model.train()
        # Freeze Image Encoder
        for param in sam_model.image_encoder.parameters():
            param.requires_grad = False
            
        epoch_loss = 0; epoch_dice = 0; epoch_prec = 0; epoch_hd = 0; steps = 0

        current_epoch = total_epochs_so_far + epoch + 1
        pbar = tqdm(train_loader, desc=f"Epoch {current_epoch}", unit="batch")

        for i, (images, masks) in enumerate(pbar):
            images = images.to(device)
            masks = masks.to(device)

            # Inputs & Upsampling
            sam_input_batch = images[:, [0, 2, 3], :, :]
            sam_input_batch_1024 = F.interpolate(sam_input_batch, size=(1024, 1024), mode='bilinear', align_corners=False)
            B = sam_input_batch_1024.shape[0]
            
            optimizer.zero_grad()
            batch_loss_accum = 0
            
            for j in range(B):
                img_single = sam_input_batch_1024[j].unsqueeze(0)
                mask_single = masks[j].unsqueeze(0).unsqueeze(0).float() 

                with torch.no_grad():
                    image_embedding = sam_model.image_encoder(img_single)

                box = torch.tensor([[0, 0, 1024, 1024]], dtype=torch.float, device=device).unsqueeze(1)
                sparse_embeddings, dense_embeddings = sam_model.prompt_encoder(points=None, boxes=box, masks=None)

                low_res_masks, _ = sam_model.mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=sam_model.prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                )

                upscaled_mask = F.interpolate(low_res_masks, size=(128, 128), mode='bilinear', align_corners=False)
                loss = loss_fn(upscaled_mask, mask_single)
                loss_norm = loss / B 
                loss_norm.backward()
                batch_loss_accum += loss.item()

                pred_prob = torch.sigmoid(upscaled_mask)
                d, p, h = calculate_metrics(pred_prob, mask_single)
                epoch_dice += d; epoch_prec += p; epoch_hd += h

            optimizer.step()
            epoch_loss += batch_loss_accum
            steps += B
            pbar.set_postfix({"Loss": f"{batch_loss_accum:.4f}"})

        # --- LOGGING ---
        print(f"Epoch {current_epoch} Summary | "
              f"Loss: {epoch_loss/(steps/B):.4f} | "
              f"Dice: {epoch_dice/steps:.4f} | "
              f"Prec: {epoch_prec/steps:.4f} | "
              f"HD: {epoch_hd/steps:.2f}")

        # --- SAVE CHECKPOINT EVERY 5 EPOCHS ---
        if current_epoch % 5 == 0:
            save_name = f"SAM_BraTS_Epoch_{current_epoch}.pth"
            torch.save(sam_model.state_dict(), save_name)
            print(f"--> Model Checkpoint saved: {save_name}")

    return sam_model

In [8]:
# Load Model Architecture
if 'sam_model' not in locals():
    # Ensure checkpoint exists
    if not os.path.exists("sam_vit_b_01ec64.pth"):
        !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
        
    sam_model = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
    sam_model.to(device)

print("=== Milestone 1: Start Training (5 Epochs) ===")
# Will save at Epoch 5
train_sam_custom(epochs_to_add=5, total_epochs_so_far=0)

print("\n=== Milestone 2: Continue Training (Total 10 Epochs) ===")
# Will save at Epoch 10
train_sam_custom(epochs_to_add=5, total_epochs_so_far=5)

print("\n=== Milestone 3: Continue Training (Total 20 Epochs) ===")
# Will save at Epoch 15 and Epoch 20
train_sam_custom(epochs_to_add=10, total_epochs_so_far=10)

# Final Save (Redundant but good for safety)
torch.save(sam_model.state_dict(), "SAM_BraTS_Final.pth")
print("\nFinal Model saved successfully!")

=== Milestone 1: Start Training (5 Epochs) ===


Epoch 1: 100%|██████████| 52/52 [12:49<00:00, 14.79s/batch, Loss=1.4146]


Epoch 1 Summary | Loss: 1.8843 | Dice: 0.0183 | Prec: 0.0653 | HD: 30.56


Epoch 2: 100%|██████████| 52/52 [12:24<00:00, 14.33s/batch, Loss=0.6656]


Epoch 2 Summary | Loss: 0.9801 | Dice: 0.2178 | Prec: 0.3487 | HD: 29.89


...

Epoch 18 Summary | Loss: -1167.9598 | Dice: 0.6414 | Prec: 0.5815 | HD: 16.16


Epoch 19: 100%|██████████| 52/52 [12:14<00:00, 14.13s/batch, Loss=-1409.3059]


Epoch 19 Summary | Loss: -1389.1448 | Dice: 0.6534 | Prec: 0.5875 | HD: 14.88


Epoch 20: 100%|██████████| 52/52 [11:57<00:00, 13.80s/batch, Loss=-2471.5125]


Epoch 20 Summary | Loss: -1606.1178 | Dice: 0.6566 | Prec: 0.5901 | HD: 14.96
--> Model Checkpoint saved: SAM_BraTS_Epoch_20.pth

Final Model saved successfully!
